In [1]:
import numpy as np

In [2]:
weights = np.array([
    [-1.8, -0.9, 0.0, 0.7, 1.5],
    [-2.4, -0.3, 0.2, 1.1, 2.0]
], dtype=np.float32)

activations = np.array([
    [0.0, 0.3, 0.8, 1.4, 2.1],
    [0.1, 0.6, 1.0, 1.8, 3.2]
], dtype=np.float32)

In [4]:
def symmetric_quantize(tensor):
    """
    Performs symmetric INT8 quantization.
    Returns:
        quantized_tensor
        scale
        zero_point
    """
    max_abs = np.max(np.abs(tensor))
    scale = max_abs / 127
    zero_point = 0
    quantized = np.round(tensor / scale)
    quantized = np.clip(quantized, -127, 127)
    return quantized.astype(np.int8), scale, zero_point

In [5]:
def asymmetric_quantize(tensor):
    """
    Performs asymmetric INT8 quantization.
    """
    x_min = np.min(tensor)
    x_max = np.max(tensor)
    if x_min == x_max:
        scale = 1.0
        zero_point = 0
    else:
        scale = (x_max - x_min) / 255
        zero_point = round(-128 - (x_min / scale))
        zero_point = int(np.clip(zero_point, -128, 127))
    quantized = np.round(tensor / scale) + zero_point
    quantized = np.clip(quantized, -128, 127)
    return quantized.astype(np.int8), scale, zero_point

In [6]:
def dequantize(quantized_tensor, scale, zero_point):
    return (quantized_tensor.astype(np.float32) - zero_point) * scale

In [7]:
def calculate_metrics(original, reconstructed, quantized):
    error = np.abs(original - reconstructed)
    mae = np.mean(error)
    mse = np.mean((original - reconstructed) ** 2)
    max_error = np.max(error)
    sat_min = np.sum(quantized == -128)
    sat_max = np.sum(quantized == 127)
    sat_total = sat_min + sat_max
    return mae, mse, max_error, sat_min, sat_max, sat_total

In [8]:
def compare_quantization(name, tensor):
    print("=" * 70)
    print(name)
    print("=" * 70)
    sym_q, sym_scale, sym_zp = symmetric_quantize(tensor)
    sym_deq = dequantize(sym_q, sym_scale, sym_zp)
    sym_metrics = calculate_metrics(tensor, sym_deq, sym_q)
    asym_q, asym_scale, asym_zp = asymmetric_quantize(tensor)
    asym_deq = dequantize(asym_q, asym_scale, asym_zp)
    asym_metrics = calculate_metrics(tensor, asym_deq, asym_q)
    print("\nOriginal Tensor\n", tensor)
    print("\nSymmetric Quantized\n", sym_q)
    print("\nSymmetric Dequantized\n", sym_deq)
    print("\nAsymmetric Quantized\n", asym_q)
    print("\nAsymmetric Dequantized\n", asym_deq)
    print("\nComparison")
    print("-" * 70)
    print(f"{'Method':15}{'Scale':12}{'ZeroPt':10}{'MAE':10}{'MSE':10}{'MaxErr':10}")
    print(f"{'Symmetric':15}{sym_scale:.6f}{sym_zp:10}{sym_metrics[0]:10.6f}{sym_metrics[1]:10.6f}{sym_metrics[2]:10.6f}")
    print(f"{'Asymmetric':15}{asym_scale:.6f}{asym_zp:10}{asym_metrics[0]:10.6f}{asym_metrics[1]:10.6f}{asym_metrics[2]:10.6f}")
    print("\nSaturation")
    print("Symmetric :", sym_metrics[3], sym_metrics[4], sym_metrics[5])
    print("Asymmetric:", asym_metrics[3], asym_metrics[4], asym_metrics[5])

In [9]:
compare_quantization("Weights", weights)
compare_quantization("Activations", activations)

Weights

Original Tensor
 [[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]

Symmetric Quantized
 [[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]

Symmetric Dequantized
 [[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]

Asymmetric Quantized
 [[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]

Asymmetric Dequantized
 [[-1.7945098  -0.8972549   0.          0.707451    1.5011765 ]
 [-2.3984313  -0.29333332  0.20705882  1.1043137   2.0015686 ]]

Comparison
----------------------------------------------------------------------
Method         Scale       ZeroPt    MAE       MSE       MaxErr    
Symmetric      0.018898         0  0.003701  0.000022  0.007874
Asymmetric     0.017255        11  0.003804  0.000021  0.007451

Saturation
Symmetric : 0 0 0
Asymmetric: 1 1 2
Activations

Original Tensor
 [[0.  0.3 0.8 1.4 2.1]
 [0.1 0.6 1.  1.8 3.2]]

Symmetric Quantized
 [[  0  12  32  56  83]
 [  4 

In [10]:
outlier_tensor = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7, 12.0],
    dtype=np.float32
)

without_outlier = np.array(
    [-0.5, -0.2, 0.0, 0.3, 0.7],
    dtype=np.float32
)

In [11]:
compare_quantization("Outlier Tensor", outlier_tensor)
compare_quantization("Without Outlier", without_outlier)

Outlier Tensor

Original Tensor
 [-0.5 -0.2  0.   0.3  0.7 12. ]

Symmetric Quantized
 [ -5  -2   0   3   7 127]

Symmetric Dequantized
 [-0.47244096 -0.18897638  0.          0.28346455  0.6614173  12.        ]

Asymmetric Quantized
 [-128 -122 -118 -112 -104  127]

Asymmetric Dequantized
 [-0.49019608 -0.19607843  0.          0.29411766  0.6862745  12.009804  ]

Comparison
----------------------------------------------------------------------
Method         Scale       ZeroPt    MAE       MSE       MaxErr    
Symmetric      0.094488         0  0.015617  0.000441  0.038583
Asymmetric     0.049020      -118  0.007190  0.000072  0.013725

Saturation
Symmetric : 0 1 1
Asymmetric: 1 1 2
Without Outlier

Original Tensor
 [-0.5 -0.2  0.   0.3  0.7]

Symmetric Quantized
 [-91 -36   0  54 127]

Symmetric Dequantized
 [-0.5015748 -0.1984252  0.         0.2976378  0.7      ]

Asymmetric Quantized
 [-128  -64  -22   42  127]

Asymmetric Dequantized
 [-0.49882355 -0.19764706  0.          0.3011765

In [12]:
def show_side_by_side(name, tensor):
    sym_q, sym_scale, sym_zp = symmetric_quantize(tensor)
    asym_q, asym_scale, asym_zp = asymmetric_quantize(tensor)
    sym_deq = dequantize(sym_q, sym_scale, sym_zp)
    asym_deq = dequantize(asym_q, asym_scale, asym_zp)
    print("\n", "="*80)
    print(name)
    print("="*80)
    print("Original\n", tensor)
    print("\nSymmetric INT8\n", sym_q)
    print("\nSymmetric Float\n", sym_deq)
    print("\nAsymmetric INT8\n", asym_q)
    print("\nAsymmetric Float\n", asym_deq)

In [13]:
show_side_by_side("Weights", weights)
show_side_by_side("Activations", activations)


Weights
Original
 [[-1.8 -0.9  0.   0.7  1.5]
 [-2.4 -0.3  0.2  1.1  2. ]]

Symmetric INT8
 [[ -95  -48    0   37   79]
 [-127  -16   11   58  106]]

Symmetric Float
 [[-1.7952756  -0.9070866   0.          0.6992126   1.4929134 ]
 [-2.4        -0.3023622   0.20787401  1.096063    2.0031495 ]]

Asymmetric INT8
 [[ -93  -41   11   52   98]
 [-128   -6   23   75  127]]

Asymmetric Float
 [[-1.7945098  -0.8972549   0.          0.707451    1.5011765 ]
 [-2.3984313  -0.29333332  0.20705882  1.1043137   2.0015686 ]]

Activations
Original
 [[0.  0.3 0.8 1.4 2.1]
 [0.1 0.6 1.  1.8 3.2]]

Symmetric INT8
 [[  0  12  32  56  83]
 [  4  24  40  71 127]]

Symmetric Float
 [[0.        0.3023622 0.8062992 1.4110236 2.0913386]
 [0.1007874 0.6047244 1.007874  1.7889764 3.2      ]]

Asymmetric INT8
 [[-128 -104  -64  -16   39]
 [-120  -80  -48   15  127]]

Asymmetric Float
 [[0.         0.30117646 0.80313724 1.4054902  2.0956862 ]
 [0.10039216 0.6023529  1.0039215  1.7945098  3.2       ]]


Outlier Tensor
Method         Scale       ZeroPt    MAE       MSE       MaxErr    
Symmetric      0.094488         0  0.015617  0.000441  0.038583
Asymmetric     0.049020      -118  0.007190  0.000072  0.013725
Without Outlier Tensor
Symmetric      0.005512         0  0.001102  0.000002  0.002362
Asymmetric     0.004706       -22  0.001176  0.000002  0.002353

Observation

Weights

Symmetric quantization worked well for the weight tensor because the values were distributed around zero. It produced low reconstruction error with a simple zero point of 0.

Activations

Asymmetric quantization performed better for the activation tensor because the values were mostly positive. The calculated zero point used the available INT8 range more efficiently.

Outlier Experiment

When the outlier value was included, the scale increased and the reconstruction error for the smaller values also increased. After removing the outlier, both quantization methods produced lower errors.